# Exercise 4.3.1 — implement an autorater

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `4.3 Interpreting Reasoning Models`  
**Notebook:** `4.3_Interpreting_Reasoning_Models_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=4.3.1](https://delta-drills.vercel.app/?arena_exercise=4.3.1)


# [4.3] Interpreting Reasoning: Thought Anchors (exercises)

> **ARENA [Streamlit Page](https://arena-chapter4-alignment-science.streamlit.app/03_[4.3]_Interpreting_Reasoning_Models)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-64.png" width="350">

# Introduction

So far in this chapter, we've focused on dividing LLM computation into small steps: single-token generation. We saw this in Indirect Object Identification where we analyzed how GPT2-Small generates a single token. But modern reasoning models produce very long **chain-of-thought** traces, so we need to think about serialized computation over many tokens, not just over layers. This requires a new abstraction.

The [Thought Anchors](https://arxiv.org/abs/2506.19143) paper introduces this abstraction by splitting reasoning traces by **sentence**. Sentences are more coherent than tokens and correspond more closely to the actual reasoning steps. Some sentences matter a lot more than others for shaping the reasoning trajectory and the final answer; the authors call these **thought anchors**. They can be identified using black-box methods (resampling rollouts) or white-box methods (looking at / intervening on attention patterns). In these exercises, we'll work through both.

<img src="https://i.snipboard.io/PBoc9G.jpg" width="700">

We'll also explore the [Thought Branches paper](https://arxiv.org/abs/2510.27484) which extends these techniques to safety-relevant scenarios like blackmail, introducing metrics like **resilience** to distinguish genuine causal drivers from post-hoc rationalizations.

## Content & Learning Objectives

### 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

### 2️⃣ Black-box Analysis

> ##### Learning Objectives
>
> - Understand **forced answer importance**: measuring impact by directly forcing model answers at specific steps
> - Implement **resampling importance**: measuring impact by having the model regenerate its own continuation from that point forward
> - Implement **counterfactual importance**: measuring impact by resampling then filtering to steps where the regenerated sentence is semantically different from the original
> - Learn when each metric is appropriate (causal vs correlational analysis)
> - Replicate key paper figures showing which reasoning steps are most critical for final answers

### 3️⃣ White-box Methods

> ##### Learning Objectives
>
> - Understand **receiver heads**: attention heads that aggregate information from reasoning steps into the final answer
> - Compute **vertical attention scores**: quantifying how much each token position attends to specific reasoning steps
> - Implement RoPE (Rotary Position Embeddings) to understand positional encoding in attention
> - Build attention suppression interventions to causally validate which attention patterns matter
> - Compare white-box attention metrics with black-box importance scores to validate mechanistic understanding

### 4️⃣ Thought Branches: Safety Applications

> ##### Learning Objectives
>
> - Apply thought anchor analysis to safety-critical scenarios (blackmail reasoning)
> - Measure how much reasoning steps influence safety-relevant outcomes (not just answer correctness)
> - Understand the **resilience metric**: how many times a sentence must be iteratively removed before it stays absent from the model's regeneration
> - Compare thought anchor patterns between mathematical reasoning (MATH benchmark) and safety reasoning (blackmail scenarios)

## Reading Material

Read at least the introduction and methods of the Thought Anchors paper before starting - the exercises follow its methodology closely. The Thought Branches paper is needed for Section 4.

- [Thought Anchors: Which LLM Reasoning Steps Matter?](https://arxiv.org/abs/2506.19143). Introduces the abstraction of treating sentences (not tokens) as units of reasoning in chain-of-thought traces, and identifies "thought anchors" - the sentences that most influence the final answer. Sections 1-3 of the exercises replicate the paper's black-box and white-box methods. Read the abstract, Section 1 (introduction) as well as 2 (explaining the methods for quantifying sentence importance) and 3 (explaining the sentence taxonomy). There's also a nice [interactive demo](https://thought-anchors.com/) showing thought anchors on example traces.
- [Thought Branches: Interpreting LLM Reasoning Requires Resampling](https://arxiv.org/abs/2510.27484). Extends thought anchor analysis to safety-relevant scenarios (e.g. blackmail reasoning), introducing the "resilience" metric. Mostly we just recommend skimming sections 1-3.
- [Principled Interpretability of Reward Hacking in Closed Frontier Models](https://www.lesswrong.com/posts/A67SbpTjuXEHK8Cvo/principled-interpretability-of-reward-hacking-in-closed) by Kroiz, Singh, Rajamanoharan & Nanda (MATS 9.0). Adapts the thought anchors resampling methodology to study reward hacking in closed frontier models (GPT-5, o3, Gemini 3 Pro), using agent actions as units of analysis rather than CoT sentences. Not needed for the exercises, but gives a sense of where the methodology is being applied.

## Setup code

Before running this code, you'll need to clone the [thought-anchors repo](https://github.com/interp-reasoning/thought-anchors). Make sure you're cloning it inside the `chapter4_alignment_science/exercises` directory.

```bash
cd chapter4_alignment_science/exercises
git clone https://github.com/interp-reasoning/thought-anchors.git
```

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter4_alignment_science"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping openai

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else repo
    if Path(repo).exists()
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import json
import math
import os
import re
import sys
import time
import warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import MethodType
from typing import Any, Callable

import circuitsvis as cv
import einops
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
import transformers
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
from IPython.display import HTML, display
from jaxtyping import Float, Int
from openai import OpenAI
from plotly.subplots import make_subplots
from scipy import stats
from sentence_transformers import SentenceTransformer
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria

warnings.filterwarnings("ignore")

thought_anchors_path = Path.cwd() / "thought-anchors"
assert thought_anchors_path.exists(), f"Please clone thought-anchors repo as {thought_anchors_path.resolve()!r}"

sys.path.append(str(thought_anchors_path))

t.set_grad_enabled(False)

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter4_alignment_science"
section = "part3_interpreting_reasoning_models"
repo = "ARENA_3.0"
root_dir = next(p for p in Path.cwd().parents if p.name == repo)
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part3_interpreting_reasoning_models.tests as tests
import part3_interpreting_reasoning_models.utils as utils

MAIN = __name__ == "__main__"

You'll need to create an `.env` file in the `chapter4_alignment_science/exercises` directory before running the next cell.

In [ ]:
# Load .env file
env_path = exercises_dir / ".env"
assert env_path.exists(), "Please create a .env file with your API keys"

load_dotenv(dotenv_path=str(env_path))

# Setup OpenRouter client (works with both Claude and OpenAI models)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Please set OPENROUTER_API_KEY in your .env file"

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

Now let's define some constants:

In [ ]:
print(f"Using device: {device}")

# 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

In this section, we'll build infrastructure to analyze chain-of-thought reasoning by breaking it into meaningful units. When models generate long reasoning traces, we need to decide what the right "units of reasoning" are.

Tokens are too granular: a single reasoning step like "Let's calculate the area: $\pi r^2$" spans multiple tokens, and splitting it apart loses semantic meaning. Sentences are a more natural unit. They correspond to complete thoughts, they're granular enough to identify specific important steps, and they align with how humans chunk reasoning.

By splitting CoT traces into sentences and categorizing them (calculations, lookups, restatements, etc.), we can ask: which types are most critical for reaching the correct answer? This sentence-level analysis sets us up for the black-box and white-box methods in later sections.

## Model Setup & Dataset Inspection

We'll start by setting a few constants (the purpose of these will become clear as we go through the exercises):

In [ ]:
# Configuration
MODEL_NAME_1B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_NAME_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
MODEL_NAME_14B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
DATASET_NAME = "uzaymacar/math-rollouts"
BLACKMAIL_DATASET_NAME = "uzaymacar/blackmail-rollouts"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.8

Next, we'll load an embedding model. Embedding models take in text and output a vector - we'll use them to measure similarity between sentences, so we can find motifs in our reasoning traces.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(embedding_model)

To understand how embedding models work, let's look at cosine similarities between a few example sentences:

In [ ]:
prompts = [
    "Wait, I think I made an error in my reasoning and need to backtrack",
    "Hold on, I believe I made a mistake in my logic and should reconsider",
    "After careful analysis, I've determined the correct answer is 42",
    "Time is an illusion. Lunchtime doubly so.",
]
labels = [x[:35] + "..." for x in prompts]

embedding = embedding_model.encode(prompts)
cosine_sims = embedding @ embedding.T

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cosine_sims, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
plt.colorbar(im, label="Cosine Similarity")
plt.title("Sentence Embedding Similarity")
plt.tight_layout()
plt.show()

We can also load the dataset that the paper's authors open-sourced. The dataset is very large, but the authors provide the structure on the [HuggingFace page](https://huggingface.co/datasets/uzaymacar/math-rollouts), so we can use `huggingface_hub` to load just the data we want. We'll inspect it more shortly.

In [ ]:
PROBLEM_ID = 4682

path = f"deepseek-r1-distill-llama-8b/temperature_0.6_top_p_0.95/correct_base_solution/problem_{PROBLEM_ID}/base_solution.json"
local_path = hf_hub_download(repo_id=DATASET_NAME, filename=path, repo_type="dataset")

with open(local_path, "r") as f:
    problem_data = json.load(f)

print("Keys in problem data:", list(problem_data.keys()))
print(f"\nProblem prompt:\n{problem_data['prompt']}")

## Sentence Splitting

First, we'll need to split our CoT traces into sentences based on punctuation and paragraph breaks. We'll also need to handle special tokens like `<think>`. We've provided the function `split_solution_into_chunks` below, which implements the following rules:

- The `<think> ... </think>` tags should be removed
- You should split on sentences (i.e. ending in any of `.`, `!`, `?`, or newlines), and characters like `:`
- You should split on periods `.` unless they are decimal numbers e.g. `x.y` or a numbered list e.g. `\n1.`
- No chunk should have length less than 10: if so, then merge it with the next chunk
- Each chunk should be stripped of whitespace

Read through the function and run the test cases below to verify it works on basic inputs.

### Recommended reading: edge cases in sentence splitting

After running the test cases, try to come up with your own input where this function breaks down or splits into chunks in a counter-intuitive way. Some things to consider: what happens with URLs, abbreviations (e.g. "Dr. Smith"), LaTeX expressions (e.g. `$3.14$`), or ellipses (`...`)? Once you've found an edge case, see if you can modify the function to handle it.

In [ ]:
def split_solution_into_chunks(text: str) -> list[str]:
    """Split solution into sentence-level chunks."""
    # Remove thinking tags
    if "<think>" in text:
        text = text.split("<think>")[1]
    if "</think>" in text:
        text = text.split("</think>")[0]
    text = text.strip()

    # Replace "." characters which we don't want to split on
    text = re.sub(r"(\d)\.(\d)", r"\1<DECIMAL>\2", text)  # e.g. "4.5" -> "4<DECIMAL>5"
    text = re.sub(r"\n(\d)\.(\s)", r"\n\1<DECIMAL>\2", text)  # e.g. "\n1. " -> "\n1<DECIMAL> "

    # Split on sentence endings, combining endings with previous chunk
    sentences = re.split(r"([!?:\n]|(?<!\n\d)\.)", text)
    chunks = []
    for i in range(0, len(sentences) - 1, 2):
        chunks.append((sentences[i] + sentences[i + 1]).replace("\n", " "))

    # Replace <DECIMAL> back with "."
    chunks = [re.sub(r"<DECIMAL>", ".", c) for c in chunks]

    # Merge chunks that are too short
    if not chunks:
        return []
    merged = [chunks[0]]
    for c in chunks[1:]:
        if len(merged[-1]) < 10:
            merged[-1] += c
        else:
            merged.append(c)
    return [c.strip() for c in merged if c.strip()]


test_cases = [
    (
        "<think>First, I understand the problem. Next, I'll solve for x. Finally, I verify!</think>",
        ["First, I understand the problem.", "Next, I'll solve for x.", "Finally, I verify!"],
    ),
    (
        "<think>Let me break this down: 1. Convert to decimal. 2. Calculate log. 3. Apply formula.</think>",
        ["Let me break this down:", "1. Convert to decimal.", "2. Calculate log.", "3. Apply formula."],
    ),
    (
        "<think>The answer is 42. Done.</think>",
        ["The answer is 42.", "Done."],
    ),
]

for input_text, expected_chunks in test_cases:
    chunks = split_solution_into_chunks(input_text)
    assert chunks == expected_chunks, f"Expected {expected_chunks}, got {chunks}"

print("All tests passed!")

## Sentence Categorization

The paper uses a taxonomy of 8 categories: Problem Setup (parsing/rephrasing the problem), Plan Generation (stating a plan of action), Fact Retrieval (recalling facts, formulas, problem details), Active Computation (algebra, calculations, manipulations), Uncertainty Management (expressing confusion, re-evaluating, backtracking), Result Consolidation (aggregating intermediate results), Self Checking (verifying previous steps), and Final Answer Emission (explicitly stating the final answer).

We define these below:

In [ ]:
CATEGORIES = {
    "problem_setup": "Problem Setup",
    "plan_generation": "Plan Generation",
    "fact_retrieval": "Fact Retrieval",
    "active_computation": "Active Computation",
    "uncertainty_management": "Uncertainty Management",
    "result_consolidation": "Result Consolidation",
    "self_checking": "Self Checking",
    "final_answer_emission": "Final Answer Emission",
    "unknown": "Unknown",
}

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "4.3.1"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part3_interpreting_reasoning_models.solutions import categorize_sentences_heuristic


### Exercise - implement an autorater

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Now we'll use a more advanced approach: an **autorater** (i.e. querying an LLM to do the categorization for us, rather than relying on hardcoded rules).

First we set up helper functions to call an API via OpenRouter (which supports many different model providers). We define `generate_response` for single calls and `generate_responses_parallel` for batch processing.

You'll need to create an `.env` file in the current working directory with `OPENROUTER_API_KEY` set, then you can run the cell below to see how the helper functions work.

In the exercise below, we've already constructed the system prompt and user prompt for you. Your job is to call `generate_responses_parallel` with the right arguments, then parse the LLM's response (a numbered list like `1. problem_setup\n2. active_computation\n...`) into a list of category labels. You should handle the case where the LLM returns more or fewer labels than expected.

In [ ]:
# These two functions are taken directly from section 4.1 (local model generation is not needed here).


def generate_response(
    model: str,
    messages: list[dict[str, str]],
    max_tokens: int = 128,
    stop_sequences: list[str] | None = None,
    temperature: float = 1.0,
    max_retries: int = 10,
) -> str:
    """Single API call with retry logic for rate limits."""
    assert openrouter_client, "OpenRouter API key not set (see earlier instructions)."

    stop_sequences = stop_sequences or []

    for attempt in range(max_retries):
        try:
            resp = openrouter_client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                stop=stop_sequences if stop_sequences else None,
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            print(str(e))
            if any(msg in str(e) for msg in ("rate_limit", "429", "empty/null choices")):
                if attempt < max_retries - 1:
                    wait_time = 2**attempt
                    print(f"Rate limit hit, waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
            raise e
    return ""


def generate_responses_parallel(
    messages_list: list[list[dict[str, str]]],
    model: str,
    max_tokens: int = 128,
    stop_sequences: list[str] | None = None,
    temperature: float = 1.0,
    max_workers: int = 10,
) -> list[str | Exception]:
    """
    Run multiple API calls in parallel using ThreadPoolExecutor.

    Args:
        messages_list: List of message lists, each to be sent as a separate API call
        model: Model identifier for OpenRouter
        max_tokens: Max tokens per response
        stop_sequences: Stop sequences for generation
        temperature: Sampling temperature
        max_workers: Maximum number of parallel workers

    Returns:
        List of responses (strings) or Exceptions for failed calls, in same order as input
    """
    results: dict[int, str | Exception] = {}
    pbar = tqdm(total=len(messages_list), desc="API calls")

    def call_single(idx: int, messages: list[dict[str, str]]) -> tuple[int, str | Exception]:
        """Helper function to make a single API call."""
        try:
            time.sleep(0.1)  # Rate limiting
            result = generate_response(model, messages, max_tokens, stop_sequences, temperature)
            return idx, result
        except Exception as e:
            return idx, e

    # Execute tasks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks with their indices
        futures = [executor.submit(call_single, i, m) for i, m in enumerate(messages_list)]

        # Process completed tasks
        for future in as_completed(futures):
            idx, result = future.result()
            results[idx] = result
            pbar.update(1)

    pbar.close()

    # Return results in original order
    return [results[i] for i in range(len(messages_list))]


# Demo of how this function works:
sys_prompt = {"role": "system", "content": "Reply in rhyming couplets."}
test_messages = [
    [sys_prompt, {"role": "user", "content": "What is 2+2?"}],
    [sys_prompt, {"role": "user", "content": "What is the capital of France?"}],
]
responses = generate_responses_parallel(test_messages, model="openai/gpt-4.1-mini", max_tokens=40)
for i, response in enumerate(responses):
    print(f"Response {i + 1}:\n{response}\n")

In [ ]:
DAG_SYSTEM_PROMPT = """
You are an expert in interpreting how language models solve math problems using multi-step reasoning. Your task is to analyze a Chain-of-Thought (CoT) reasoning trace, broken into discrete text chunks, and label each chunk with a tag that describes what this chunk is *doing* functionally in the reasoning process.

### Function Tags:

1. `problem_setup`: Parsing or rephrasing the problem
2. `plan_generation`: Stating or deciding on a plan of action
3. `fact_retrieval`: Recalling facts, formulas, problem details
4. `active_computation`: Performing algebra, calculations, manipulations
5. `result_consolidation`: Aggregating intermediate results, summarizing
6. `uncertainty_management`: Expressing confusion, re-evaluating, backtracking
7. `final_answer_emission`: Explicit statement of the final boxed answer
8. `self_checking`: Verifying previous steps
9. `unknown`: Use only if the chunk does not fit any of the above

### Output Format:

Return a numbered list with one function tag per chunk:

1. problem_setup
2. active_computation
...
"""


def categorize_sentences_autorater(
    problem_text: str, chunks: list[str], model: str = "openai/gpt-4.1-mini"
) -> list[str]:
    """Categorize sentences using an LLM autorater."""

    chunk_str = "\n".join(f"{i + 1}. {chunk}" for i, chunk in enumerate(chunks))

    user_prompt = f"""
Here is the math problem:

[PROBLEM]
{problem_text}

Here is the Chain of Thought, broken into chunks:

[CHUNKS]
{chunk_str}

Now label each chunk with function tags."""

    messages_list = [
        [
            {"role": "system", "content": DAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    ]

    # YOUR CODE HERE - get results with `generate_responses_parallel`, and parse them into
    # a list of category labels for each chunk.


example_problem = "What is the area of a circle with radius 5?"
autorater_categories = categorize_sentences_autorater(example_problem, example_sentences)

# Lightweight verification
assert len(autorater_categories) == len(example_sentences), (
    f"Expected {len(example_sentences)} categories, got {len(autorater_categories)}"
)
assert all(cat in CATEGORIES.values() or cat == "Unknown" for cat in autorater_categories), (
    f"All categories should be valid, got: {autorater_categories}"
)

df = pd.DataFrame(
    {
        "Sentence": example_sentences,
        "Heuristic": categories,
        "Autorater": autorater_categories,
    }
)
display(df)

<details><summary>Solution</summary>

```python
DAG_SYSTEM_PROMPT = """
You are an expert in interpreting how language models solve math problems using multi-step reasoning. Your task is to analyze a Chain-of-Thought (CoT) reasoning trace, broken into discrete text chunks, and label each chunk with a tag that describes what this chunk is *doing* functionally in the reasoning process.

### Function Tags:

1. `problem_setup`: Parsing or rephrasing the problem
2. `plan_generation`: Stating or deciding on a plan of action
3. `fact_retrieval`: Recalling facts, formulas, problem details
4. `active_computation`: Performing algebra, calculations, manipulations
5. `result_consolidation`: Aggregating intermediate results, summarizing
6. `uncertainty_management`: Expressing confusion, re-evaluating, backtracking
7. `final_answer_emission`: Explicit statement of the final boxed answer
8. `self_checking`: Verifying previous steps
9. `unknown`: Use only if the chunk does not fit any of the above

### Output Format:

Return a numbered list with one function tag per chunk:

1. problem_setup
2. active_computation
...
"""


def categorize_sentences_autorater(
    problem_text: str, chunks: list[str], model: str = "openai/gpt-4.1-mini"
) -> list[str]:
    """Categorize sentences using an LLM autorater."""

    chunk_str = "\n".join(f"{i + 1}. {chunk}" for i, chunk in enumerate(chunks))

    user_prompt = f"""
Here is the math problem:

[PROBLEM]
{problem_text}

Here is the Chain of Thought, broken into chunks:

[CHUNKS]
{chunk_str}

Now label each chunk with function tags."""

    messages_list = [
        [
            {"role": "system", "content": DAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    ]

    responses = generate_responses_parallel(
        messages_list=messages_list,
        model=model,
        max_tokens=128,
        temperature=0.0,
    )

    raw_response = responses[0] if isinstance(responses[0], str) else ""
    response = re.split(r"\n\d{1,2}\.", "\n" + raw_response.strip())
    response = [r.strip() for r in response if r.strip()]

    if len(response) != len(chunks):
        print(f"Warning: length mismatch {len(response)} != {len(chunks)}")
        response = response[: len(chunks)] + ["unknown"] * (len(chunks) - len(response))

    return [CATEGORIES.get(r, "Unknown") for r in response]


example_problem = "What is the area of a circle with radius 5?"
autorater_categories = categorize_sentences_autorater(example_problem, example_sentences)

# Lightweight verification
assert len(autorater_categories) == len(example_sentences), (
    f"Expected {len(example_sentences)} categories, got {len(autorater_categories)}"
)
assert all(cat in CATEGORIES.values() or cat == "Unknown" for cat in autorater_categories), (
    f"All categories should be valid, got: {autorater_categories}"
)

df = pd.DataFrame(
    {
        "Sentence": example_sentences,
        "Heuristic": categories,
        "Autorater": autorater_categories,
    }
)
display(df)
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
